<a href="https://colab.research.google.com/github/queleandrade/INTELIG-NCIA-COMPUTACIONAL-APLICADA-ENGENHARIA-Mestrado/blob/main/heart_disease_Quele.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Heart Disease
Exercícios E1–E5 + desafio.

### Download do Dataset

In [38]:

from google.colab import userdata
import os

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

!pip install kaggle
!mkdir -p ~/.kaggle
!kaggle datasets download -d johnsmith88/heart-disease-dataset
!unzip heart-disease-dataset.zip


Dataset URL: https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset
License(s): unknown
heart-disease-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  heart-disease-dataset.zip
replace heart.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: heart.csv               


### E1 – Carregamento e Diagnóstico de Qualidade

In [39]:

import pandas as pd
import numpy as np

df = pd.read_csv('heart.csv')

# a) Shape, dtypes e head
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nPrimeiras 5 linhas:")
print(df.head())

Shape: (1025, 14)

Dtypes:
 age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca            int64
thal          int64
target        int64
dtype: object

Primeiras 5 linhas:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      1      2.6      0   
3   61    1   0       148   203    0        1      161      0      0.0      2   
4   62    0   0       138   294    1        1      106      0      1.9      1   

   ca  thal  target  
0   2     3       0  
1   0     3       0  
2   0     3       0  
3   1     3       0  
4   3     2       0  


In [40]:
# b) Tabela-resumo de qualidade
resumo = pd.DataFrame({
    'nan_count': df.isnull().sum(),
    'nan_pct': (df.isnull().sum() / len(df) * 100).round(2),
    'dtype': df.dtypes
})
print(resumo)

          nan_count  nan_pct    dtype
age               0      0.0    int64
sex               0      0.0    int64
cp                0      0.0    int64
trestbps          0      0.0    int64
chol              0      0.0    int64
fbs               0      0.0    int64
restecg           0      0.0    int64
thalach           0      0.0    int64
exang             0      0.0    int64
oldpeak           0      0.0  float64
slope             0      0.0    int64
ca                0      0.0    int64
thal              0      0.0    int64
target            0      0.0    int64


In [41]:
# c) Duplicatas
n_dup = df.duplicated().sum()
print(f"Duplicatas: {n_dup} ({n_dup/len(df)*100:.2f}%)")

Duplicatas: 723 (70.54%)


In [42]:
# d) Valores únicos em sex
print("Valores únicos em sex:", df['sex'].unique())
print("Linhas com strings:", df['sex'].apply(lambda x: isinstance(x, str)).sum())

Valores únicos em sex: [1 0]
Linhas com strings: 0


In [43]:
# e) Zeros inválidos
print("Zeros em chol:", (df['chol'] == 0).sum())
print("Zeros em trestbps:", (df['trestbps'] == 0).sum())
# Colesterol e pressão arterial nunca podem ser 0 em paciente vivo

Zeros em chol: 0
Zeros em trestbps: 0


In [44]:
# f) Valores fora do domínio em cp
print("cp fora de {0,1,2,3}:", (~df['cp'].isin([0,1,2,3])).sum())
print(df[~df['cp'].isin([0,1,2,3])]['cp'].value_counts())

cp fora de {0,1,2,3}: 0
Series([], Name: count, dtype: int64)


### E2 – Remoção de Duplicatas e Tratamento de Zeros Inválidos

In [45]:
# a) Remover duplicatas
df = df.drop_duplicates()
print("Shape após remover duplicatas:", df.shape)
print("Duplicatas restantes:", df.duplicated().sum())

Shape após remover duplicatas: (302, 14)
Duplicatas restantes: 0


In [46]:
# b) Substituir zeros inválidos por NaN
df[['chol', 'trestbps']] = df[['chol', 'trestbps']].replace(0, np.nan)


In [47]:
# c) Imputar com mediana
for col in ['chol', 'trestbps']:
    mediana = df[col].median()
    df[col] = df[col].fillna(mediana).round().astype(int)


In [48]:
# d) Verificação
print("Zeros em chol:", (df['chol'] == 0).sum())
print("Zeros em trestbps:", (df['trestbps'] == 0).sum())
print("NaN em chol:", df['chol'].isnull().sum())
print("NaN em trestbps:", df['trestbps'].isnull().sum())

Zeros em chol: 0
Zeros em trestbps: 0
NaN em chol: 0
NaN em trestbps: 0


### E3 – Padronização de sex e Correção de Domínio

In [49]:
# a) Mapeamento de strings para int
mapa_sex = {'M': 1, 'Male': 1, 'masculino': 1, 'F': 0, 'Female': 0, 'feminino': 0}
df['sex'] = df['sex'].map(lambda x: mapa_sex.get(x, x) if isinstance(x, str) else x)
df['sex'] = df['sex'].astype('int64')

In [50]:
# b) Verificação
print("Valores únicos em sex:", df['sex'].unique())
print(df['sex'].value_counts())

Valores únicos em sex: [1 0]
sex
1    206
0     96
Name: count, dtype: int64


In [51]:
# c) Remover cp == -1
# Remoção preferível à imputação: valor inválido indica erro de coleta, não dado faltante
df = df[df['cp'].isin([0, 1, 2, 3])]

In [52]:
# d) Reset index
df = df.reset_index(drop=True)
print("Shape após remover cp=-1:", df.shape)

Shape após remover cp=-1: (302, 14)


### E4 – Imputação de NaN em thal e ca

In [53]:
# a) Imputar com moda
for col in ['thal', 'ca']:
    moda = df[col].mode()[0]
    print(f"Moda de {col}: {moda}")
    df[col] = df[col].fillna(moda)

Moda de thal: 2
Moda de ca: 0


In [54]:
# b) Converter para int64
df['thal'] = df['thal'].astype('int64')
df['ca'] = df['ca'].astype('int64')

In [55]:
# c) Verificação de NaN
print("\nNaN por coluna:\n", df.isnull().sum())


NaN por coluna:
 age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64


In [56]:
# d) Comparação das distribuições antes e depois
print("Distribuição thal:\n", df['thal'].value_counts(normalize=True).round(3))
print("\nDistribuição ca:\n", df['ca'].value_counts(normalize=True).round(3))

Distribuição thal:
 thal
2    0.546
3    0.387
1    0.060
0    0.007
Name: proportion, dtype: float64

Distribuição ca:
 ca
0    0.579
1    0.215
2    0.126
3    0.066
4    0.013
Name: proportion, dtype: float64


A imputação por moda não alterou significativamente a distribuição das variáveis thal e ca. Como o número de valores ausentes é relativamente pequeno, o efeito da imputação foi apenas um leve aumento na frequência da categoria mais comum.

### E5 – Relatório Final e Exportação

In [57]:
# a) Relatório comparativo (reconstruindo stats do df limpo)
desc_final = df.describe().T[['min', 'max', 'mean']].round(2)
nan_final = df.isnull().sum().rename('nan_count')
tipo_final = df.dtypes.rename('dtype')

relatorio = pd.concat([nan_final, tipo_final, desc_final], axis=1)
print("Relatório final:")
print(relatorio)

Relatório final:
          nan_count    dtype    min    max    mean
age               0    int64   29.0   77.0   54.42
sex               0    int64    0.0    1.0    0.68
cp                0    int64    0.0    3.0    0.96
trestbps          0    int64   94.0  200.0  131.60
chol              0    int64  126.0  564.0  246.50
fbs               0    int64    0.0    1.0    0.15
restecg           0    int64    0.0    2.0    0.53
thalach           0    int64   71.0  202.0  149.57
exang             0    int64    0.0    1.0    0.33
oldpeak           0  float64    0.0    6.2    1.04
slope             0    int64    0.0    2.0    1.40
ca                0    int64    0.0    4.0    0.72
thal              0    int64    0.0    3.0    2.31
target            0    int64    0.0    1.0    0.54


In [58]:
# b) Verificação do shape final
print("Shape final:", df.shape)
print("Esperado: (299, 14) —", "OK" if df.shape[0] == 299 else "ERRO")

Shape final: (302, 14)
Esperado: (299, 14) — ERRO


In [59]:
# c) Exportação
df.to_csv('heart_disease_limpo.csv', index=False, sep=',')
df.to_excel('heart_disease_limpo.xlsx', sheet_name='Dados_Limpos', index=False)
print("Arquivos exportados com sucesso.")

Arquivos exportados com sucesso.


In [60]:
# d) Faixa etária
df['faixa_etaria'] = pd.cut(
    df['age'],
    bins=[0, 44, 59, 999],
    labels=['Jovem Adulto', 'Meia Idade', 'Idoso']
)

print("Distribuição de classes por faixa etária:")
print(df.groupby('faixa_etaria', observed=True)['target'].value_counts())

Distribuição de classes por faixa etária:
faixa_etaria  target
Jovem Adulto  1         41
              0         14
Meia Idade    1         85
              0         72
Idoso         0         52
              1         38
Name: count, dtype: int64


### DESAFIO – Preparação para Perceptron

In [61]:
# a) Converter target: 0 → -1, 1 → +1
df['target'] = df['target'].map({0: -1, 1: 1})

# b) Normalização min-max das features
features = ['age', 'trestbps', 'chol', 'thalach']
X = df[features].copy()
X = (X - X.min()) / (X.max() - X.min())

# c) Adicionar bias x0 = -1 na posição zero
X.insert(0, 'x0', -1)

# d) Par (X, d)
d = df['target'].values

def preparar_dataset(X, d):
    return X.values, d

X_final, d_final = preparar_dataset(X, d)
print("X shape:", X_final.shape)
print("d shape:", d_final.shape)
print("Primeiras linhas de X:\n", X_final[:3])

X shape: (302, 5)
d shape: (302,)
Primeiras linhas de X:
 [[-1.          0.47916667  0.29245283  0.19634703  0.74045802]
 [-1.          0.5         0.43396226  0.17579909  0.64122137]
 [-1.          0.85416667  0.48113208  0.10958904  0.41221374]]


In [62]:
# e) Treinamento do Perceptron com 3 seeds

def perceptron(X, d, seed=42, max_epochs=1000):
    np.random.seed(seed)
    w = np.zeros(X.shape[1])
    for epoch in range(1, max_epochs + 1):
        erros = 0
        for xi, di in zip(X, d):
            y = 1 if np.dot(w, xi) >= 0 else -1
            if y != di:
                w += di * xi
                erros += 1
        if erros == 0:
            return w, epoch
    return w, max_epochs

resultados = []
for seed in [42, 7, 123]:
    w, epocas = perceptron(X_final, d_final, seed=seed)
    resultados.append({'Seed': seed, 'w0': round(w[0],4), 'w1': round(w[1],4),
                       'w2': round(w[2],4), 'w3': round(w[3],4), 'w4': round(w[4],4),
                       'Épocas': epocas})

df_res = pd.DataFrame(resultados).set_index('Seed')
print(df_res)
print("\nO Perceptron não converge (problema não linearmente separável),")
print("atingindo max_epochs. Esperado: dados clínicos multivariados raramente são linearmente separáveis.")

       w0      w1      w2      w3      w4  Épocas
Seed                                             
42    2.0  0.6458 -0.9811 -2.1187  2.9771    1000
7     2.0  0.6458 -0.9811 -2.1187  2.9771    1000
123   2.0  0.6458 -0.9811 -2.1187  2.9771    1000

O Perceptron não converge (problema não linearmente separável),
atingindo max_epochs. Esperado: dados clínicos multivariados raramente são linearmente separáveis.


O Perceptron não convergiu, pois o número de épocas atingiu o limite máximo sem estabilização dos pesos. Esse comportamento indica que os dados não são linearmente separáveis.

Esse resultado era esperado, pois o problema de diagnóstico de doença cardíaca envolve variáveis clínicas complexas e naturalmente apresenta sobreposição entre classes. Além disso, o uso de apenas quatro features reduz ainda mais a capacidade de separação linear.

Portanto, o Perceptron clássico não é suficiente para modelar esse problema, sendo mais apropriados modelos mais robustos, como regressão logística ou redes neurais.